## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Pose estimation - Application, part 2: gait videos in cerebral palsy

In part 1, we had only 89 videos, and the movement measures added little information to age, sex and BMI. Here we use a much larger clinical dataset.

>Kidziński, Ł., Yang, B., Hicks, J. L., Rajagopal, A., Delp, S. L., & Schwartz, M. H. (2020). Deep neural networks enable quantitative movement analysis using single-camera videos. *Nature Communications*, 11, 4054.

You already know this study: in lecture 04.03 we used its **motion-capture** gait parameters to classify GMFCS levels. The same children and young adults with cerebral palsy (CP) were also recorded with a **normal video camera** while they walked. The authors ran the pose estimation model **OpenPose** on each video and shared the keypoints (not the videos, to protect the privacy of the patients).

So we can ask a clinically important question: **can a normal video camera give the same information as a motion-capture laboratory?**

The data is in `data/cp_video_gait` (see the `README.md` in that folder):
- `cp_video_features.csv`: one row per video (3,258 videos, 1,374 patients), with the clinical labels, the demographics, and movement features that we computed from the keypoints
- `cp_video_trajectories.npz`: the keypoint trajectories of 1,000 videos (one video per patient)

The license of the data is CC BY-NC 2.0: non-commercial use, with attribution.

In this notebook we look at the keypoints and see how the features were computed. In part 3 we build the models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, medfilt

z = np.load('data/cp_video_gait/cp_video_trajectories.npz')
X = z['keypoints'].astype(float)          # (videos, frames, joints, 2)
joint_names = [str(n) for n in z['joint_names']]
fps = float(z['fps'])
J = {name: i for i, name in enumerate(joint_names)}

features = pd.read_csv('data/cp_video_gait/cp_video_features.csv')
print('Trajectories:', X.shape, '(videos, frames, joints, x/y)')
print('Joints:', joint_names)
print('Feature table:', features.shape, '| patients:', features.patient_id.nunique())

The trajectories use **13 joints** from the OpenPose BODY_25 format. The frame rate here is 15 frames/s (we kept every second frame; this is an estimate, see the README).

## The labels

- **GDI (Gait Deviation Index)**: one number that summarizes how much the joint angles during walking (from motion capture) differ from typical gait. 100 = typical gait; each 10 points below 100 = one standard deviation away from typical. This will be our main target.
- **GMFCS level**: the mobility level that you used in 04.03 (1 to 4; the codes 0 and 6 are not valid levels, so we do not use them for GMFCS).
- `speed` and `cadence`: measured in the motion-capture laboratory.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(features.GDI, bins=40)
ax[0].axvline(100, color='k', linestyle='--'); ax[0].set_xlabel('GDI'); ax[0].set_title('GDI of all videos')
valid = features[features.gmfcs.between(1, 4)]
valid.boxplot(column='GDI', by='gmfcs', ax=ax[1]); ax[1].set_title('GDI by GMFCS level'); ax[1].set_xlabel('GMFCS')
ax[2].scatter(features.speed, features.GDI, s=5, alpha=0.4)
ax[2].set_xlabel('Speed, motion capture (m/s)'); ax[2].set_ylabel('GDI'); ax[2].set_title('GDI vs speed')
plt.suptitle('')
plt.tight_layout()
plt.show()

print('Videos per patient:', features.groupby('patient_id').size().value_counts().sort_index().to_dict())

Most patients have a GDI below 100 (their gait is not typical). Higher GMFCS levels have a lower GDI, and slower patients have a lower GDI. Many patients came for more than one visit, so in part 3 we must split the data **by patient** (lecture 04.04).

## Look at the keypoints

Let us plot the skeleton of one video in a few frames.

In [ ]:
bones = [('neck', 'mid_hip'), ('r_shoulder', 'l_shoulder'), ('neck', 'nose'),
         ('mid_hip', 'r_hip'), ('r_hip', 'r_knee'), ('r_knee', 'r_ankle'), ('r_ankle', 'r_big_toe'),
         ('mid_hip', 'l_hip'), ('l_hip', 'l_knee'), ('l_knee', 'l_ankle'), ('l_ankle', 'l_big_toe')]

def plot_skeleton(ax, k, title=''):
    for a, b in bones:
        color = 'tab:red' if a.startswith('r_') or b.startswith('r_') else 'tab:blue'
        ax.plot(k[[J[a], J[b]], 0], k[[J[a], J[b]], 1], color=color, linewidth=2)
    ax.invert_yaxis(); ax.set_aspect('equal'); ax.set_title(title)

meta = features.set_index('video_id').loc[z['video_id']].reset_index()
meta['video_id'] = meta['video_id'].astype(int)
i = int(np.argmax((meta.GDI > 95) & (np.isfinite(X[:, :, J['mid_hip'], 0]).mean(axis=1) > 0.8)))   # a video with almost typical gait
fig, ax = plt.subplots(1, 5, figsize=(16, 4))
for a, f in zip(ax, [60, 90, 120, 150, 180]):
    plot_skeleton(a, X[i, f], f'frame {f}')
plt.suptitle(f"Video {meta.video_id[i]} (GDI = {meta.GDI[i]:.0f}). Red: right side, blue: left side")
plt.show()

The videos were recorded from the side and from the front, and the camera can move to follow the patient. So the position of the patient in the image changes, and the size changes when the patient walks toward the camera. This is one reason why we use **angles** and **ratios** (lengths divided by the leg length), which do not depend on the position or the size.

## From keypoints to a knee angle

The 2D knee angle is the angle at the knee between the thigh (knee → hip) and the shank (knee → ankle). 180 degrees = straight leg.

In [ ]:
def fill_gaps(x):
    x = np.asarray(x, dtype=float).copy(); ok = np.isfinite(x)
    if ok.sum() < 10: return x
    x[~ok] = np.interp(np.flatnonzero(~ok), np.flatnonzero(ok), x[ok])
    return x

def joint_xy(v, name):
    """Trajectory of one joint (frames, 2), gaps filled and smoothed."""
    xy = X[v, :, J[name], :]
    return np.stack([savgol_filter(medfilt(fill_gaps(xy[:, d]), 5), 7, 2) for d in range(2)], axis=1)

def angle(a, b, c):
    """Angle at point b (degrees) between the lines b->a and b->c, for every frame."""
    v1, v2 = a - b, c - b
    cos = (v1 * v2).sum(1) / (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def knee_angle(v, side='r'):
    return angle(joint_xy(v, f'{side}_hip'), joint_xy(v, f'{side}_knee'), joint_xy(v, f'{side}_ankle'))

# compare a patient with almost typical gait and a patient with a low GDI
good = i
quality = np.isfinite(X[:, :, J['mid_hip'], 0]).mean(axis=1) > 0.8
low = int(np.argmax((meta.GDI < 55) & quality))
t = np.arange(X.shape[1]) / fps

plt.figure(figsize=(12, 4))
for v, label in [(good, 'high GDI'), (low, 'low GDI')]:
    plt.plot(t, knee_angle(v), label=f'{label} ({meta.GDI[v]:.0f}), video {meta.video_id[v]}')
plt.xlabel('Time (s, estimated)'); plt.ylabel('Right knee angle (deg)'); plt.legend()
plt.title('2D knee angle during walking')
plt.show()

In the patient with a high GDI, the knee angle changes a lot in each step (the knee bends and straightens). In the patient with a low GDI, the range is smaller, and the knee stays more bent (the angle stays further from 180). This is typical of crouch gait in CP.

The flat parts at the start are frames with no detection, filled by interpolation. Some parts of the signal are noisy: when the patient turns, or walks toward the camera, the 2D angle is not a good measure of the true knee flexion.

## The features

For each video, we summarized the trajectories with a few numbers (see the README for the full list). For example, the knee features are the 5th and 95th percentiles of the knee angle, and the range between them. Let us compute the knee range for the two patients, and compare it with the value in the feature table. (The values are not exactly the same: the table was computed from the full frame rate.)

In [ ]:
for v in [good, low]:
    k = knee_angle(v)
    print(f"video {meta.video_id[v]}: knee range from the trajectory = {np.nanpercentile(k, 95) - np.nanpercentile(k, 5):5.1f} deg | "
          f"in the table = {meta.knee_angle_range_R[v]:5.1f} deg")

The feature table has these groups of columns:

In [ ]:
id_cols = ['video_id', 'patient_id', 'exam_id', 'diagnosis']
label_cols = ['gmfcs', 'GDI']
demo_cols = ['age', 'height', 'mass', 'bmi']
lab_cols = ['speed', 'cadence']
video_cols = [c for c in features.columns if c not in id_cols + label_cols + demo_cols + lab_cols]
print('Demographics:  ', demo_cols)
print('Motion capture:', lab_cols)
print(f'Video features ({len(video_cols)}):', video_cols)
features[video_cols].describe().T[['mean', 'std', 'min', 'max']].round(2)

**Questions to think about**
- Which video features do you expect to be related to GDI? Why?
- The keypoints come from a normal camera, at about 15 to 30 frames/s, with no markers on the body. What limits do you expect, compared with motion capture?

In part 3, we test whether these video features can predict GDI and GMFCS, and we compare them with the demographics and with the motion-capture measures.